In [1]:
from pathlib import Path
import json
import random

ROOT = Path.cwd().resolve()

SOURCE_DATA = ROOT.parent / "data"

DATA = ROOT / "data"
IMG = ROOT / "img"

DATA.mkdir(parents=True, exist_ok=True)
IMG.mkdir(parents=True, exist_ok=True)

SOURCE = SOURCE_DATA / "fresh_2026.jsonl"

assert SOURCE.exists(), f"Файл не найден: {SOURCE}"

print("Рабочая папка:", ROOT)
print("Исходный файл:", SOURCE)
print("Результаты:", DATA)
print("Графики:", IMG)

Рабочая папка: C:\Users\alex-\Documents\AI-agent\pythonProject4\hw_3
Исходный файл: C:\Users\alex-\Documents\AI-agent\pythonProject4\data\fresh_2026.jsonl
Результаты: C:\Users\alex-\Documents\AI-agent\pythonProject4\hw_3\data
Графики: C:\Users\alex-\Documents\AI-agent\pythonProject4\hw_3\img


In [52]:
import os, re, gzip, time, warnings
os.environ.setdefault("LANGCHAIN_OPENAI_TCP_KEEPALIVE", "0")
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from typing import TypedDict
import httpx
import difflib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()

KEY = os.getenv("OPENROUTER_API_KEY")
BASE_URL = "https://openrouter.ai/api/v1"
CHEAP, MID, STRONG = "openai/gpt-4o-mini", "anthropic/claude-haiku-4.5", "anthropic/claude-sonnet-4.6"
COLORS = {"violet": "#5436A3", "amber": "#F09000", "teal": "#00838F", "red": "#C43C3C", "grey": "#787882"}
SAMPLE = 40
DRY_RUN = False

def load_jsonl(name):
    return [json.loads(line) for line in (DATA / name).read_text(encoding="utf-8").splitlines() if line.strip()]

def pmap(fn, items, workers=1):
    if workers == 1: return [fn(item) for item in items]
    with ThreadPoolExecutor(workers) as ex: return list(ex.map(fn, items))

In [4]:
SEED = 42
SAMPLE_SIZE = 40

source = [
    json.loads(s)
    for s in SOURCE.read_text(encoding="utf-8").splitlines()
    if s.strip()
]

wiki_questions = [
    q for q in source
    if str(q.get("id", "")).startswith("fresh-")
]

assert len(wiki_questions) >= SAMPLE_SIZE, (
    f"Недостаточно вопросов: {len(wiki_questions)}"
)

pool = wiki_questions

indices = sorted(
    random.Random(SEED).sample(
        range(len(pool)),
        SAMPLE_SIZE
    )
)

tasks = [pool[i] for i in indices]

assert len(tasks) == SAMPLE_SIZE
assert len({q["id"] for q in tasks}) == SAMPLE_SIZE

def save_jsonl(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

save_jsonl(DATA / "wikipedia_pool.jsonl", pool)
save_jsonl(DATA / "fresh_2026.jsonl", tasks)

manifest = {
    "source": str(SOURCE),
    "seed": SEED,
    "pool_size": len(pool),
    "sample_size": SAMPLE_SIZE,
    "indices": indices,
    "selected_ids": [q["id"] for q in tasks]
}

(DATA / "selection_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Всего в исходном файле:", len(source))
print("Wikipedia-вопросов:", len(wiki_questions))
print("Выбрано для ДЗ №3:", len(tasks))
print("Первые ID:", [q["id"] for q in tasks[:10]])

Всего в исходном файле: 76
Wikipedia-вопросов: 56
Выбрано для ДЗ №3: 40
Первые ID: ['fresh-0', 'fresh-2', 'fresh-3', 'fresh-4', 'fresh-5', 'fresh-7', 'fresh-8', 'fresh-9', 'fresh-10', 'fresh-15']


## Шаг 1. Счетчик и базовая линия

In [19]:
CALLS = globals().get("CALLS", [])
FIRST = globals().get("FIRST", {})
FIRST_RAW = globals().get("FIRST_RAW", {})
TAG = globals().get("TAG", {"now": "без метки"})

In [20]:
def on_response(response):
    response.read()

    try:
        request = json.loads(
            response.request.content or b"{}"
        )
        payload = response.json()

        usage = payload.get("usage") or {}

    except (ValueError, TypeError):
        return

    record = {
        "tag": TAG["now"],
        "model": request.get("model"),
        "messages": len(request.get("messages", [])),
        "tools": len(request.get("tools", [])),
        "prompt": usage.get("prompt_tokens"),
        "completion": usage.get("completion_tokens"),
        "cost": usage.get("cost"),
        "status": response.status_code,
        "has_usage": isinstance(
            payload.get("usage"), dict
        )
    }

    CALLS.append(record)

    FIRST.setdefault(
        TAG["now"],
        request
    )
    FIRST_RAW.setdefault(
    TAG["now"],
    bytes(response.request.content)
    )

    with (DATA / "http_calls.jsonl").open(
        "a", encoding="utf-8"
    ) as fp:
        fp.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

In [21]:
try:
    HTTP.close()
except NameError:
    pass

HTTP = httpx.Client(
    event_hooks={"response": [on_response]},
    timeout=httpx.Timeout(
        connect=10,
        read=40,
        write=10,
        pool=10
    )
)

print("HTTP hooks:", len(HTTP.event_hooks["response"]))

HEADERS = {"Authorization": f"Bearer {KEY}" if KEY else "", "X-Title": "agents-course-homework03"}

HTTP hooks: 1


In [22]:
def chat(messages, model=CHEAP, tools=None, temperature=0.0):
    body = {
        "model": model,
        "messages": messages,
        "temperature": temperature
    }

    if tools:
        body["tools"] = tools

    for attempt in range(2):
        try:
            r = HTTP.post(
                f"{BASE_URL}/chat/completions",
                json=body,
                headers=HEADERS
            )
        except httpx.RequestError as e:
            if attempt == 1:
                raise RuntimeError(f"LLM network error: {e}") from e
            continue

        if r.status_code == 200:
            return r.json()["choices"][0]["message"]

        if r.status_code not in (429, 500, 502, 503, 504):
            break

        if attempt == 0:
            time.sleep(2)

    raise RuntimeError(f"LLM HTTP {r.status_code}: {r.text[:200]}")

In [23]:
WIKI = "https://en.wikipedia.org/w/api.php"
WIKI_HEADERS = {"User-Agent": "agents-course-homework03/1.0"}
SNAPSHOT = DATA / "wiki_snapshot.json.gz"
WIKI_CACHE = json.loads(gzip.decompress(SNAPSHOT.read_bytes())) if SNAPSHOT.exists() else {}


In [24]:
OFFLINE_WIKI = True

CACHE_MISSES = []
CACHE_HITS = []

def wiki(params):
    key = json.dumps(params, sort_keys=True)

    if key in WIKI_CACHE:
        CACHE_HITS.append(key)
        return WIKI_CACHE[key]

    CACHE_MISSES.append(key)

    if OFFLINE_WIKI:
        if params.get("list") == "search":
            return {"search": []}
        if params.get("prop") == "extracts":
            return {"pages": {}}
        return {}


def web_search(query: str) -> str:
    """Search English Wikipedia: returns the intro of the best article and the titles of two more articles."""
    hits = wiki({'action': 'query', 'list': 'search', 'srsearch': query, 'srlimit': 3}).get('search', [])
    if not hits:
        return 'nothing found'
    pages = wiki({'action': 'query', 'prop': 'extracts', "explaintext": 1, "exintro": 1, "titles": hits[0]["title"]}).get("pages", {})
    intro = " ".join(next(iter(pages.values()), {}).get('extract', '').split())[:1200]
    return f"[{hits[0]['title']}] {intro} | other articles:" + ", ".join(h['title'] for h in hits[1:])

def page_find(title: str, keywords: str) -> str:
    """Look inside the full text of a Wikipedia article."""

    pages = wiki({
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": title,
        "redirects": 1
    }).get("pages", {})

    text = next(
        iter(pages.values()), {}
    ).get("extract", "")

    if not text:
        return "no such page"

    kws = [
        w for w in re.findall(r"\w+", keywords.lower())
        if len(w) > 2
    ]

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
        and sum(w in line.lower() for w in kws)
        >= max(1, (len(kws) + 1) // 2)
    ]

    return (
        "\n".join(line[:350] for line in lines[:5])
        or "keywords not found on the page"
    )

print("Wikipedia tools ready; cached queries:", len(WIKI_CACHE))

Wikipedia tools ready; cached queries: 1640


In [25]:
AGENT_SYSTEM = ("You answer questions about events of 2026 using Wikipedia tools. Never answer from memory, do not repeat the same call. "
                "When done, reply with the short answer only, no explanation.")
BARE_TOOLS = [
    {"type": "function", "function": {"name": "web_search", "description": web_search.__doc__,
     "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "short query: a name, a title, an event"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "page_find", "description": page_find.__doc__,
     "parameters": {"type": "object", "properties": {"title": {"type": "string", "description": "exact article title, e.g. 2026 in Japan"},
                                                     "keywords": {"type": "string", "description": "two to five keywords"}},
                    "required": ["title", "keywords"]}}}]
FUNCS = {"web_search": web_search, "page_find": page_find}
LAST_STEP = "No more tool calls. Answer now with what you found."


In [26]:
def bare_agent(question, model=CHEAP, max_steps=6):
    messages, seen = [{'role': 'system', 'content': AGENT_SYSTEM},{"role": "user", "content": question}], set()
    for step in range(max_steps):
        last = step == max_steps - 1
        if last:
            messages.append({'role': 'user', 'content': LAST_STEP})
        msg = chat(messages, model, None if last else BARE_TOOLS)
        messages.append({k: v for k, v in msg.items() if k in ('role', 'content', 'tool_calls') and v})
        calls = msg.get('tool_calls') or []
        if not calls:
            return msg.get('content') or ''
        for c in calls:
            key = (c['function']['name'], c['function']['arguments'])
            result = 'repeated call, answer now' if key in seen else FUNCS[key[0]](**json.loads(key[1]))
            seen.add(key)
            messages.append({"role": "tool", "tool_call_id": c["id"], "content": result})
    return messages[-1].get("content") or ""

немного подкорректируем проверку правильности и устойчивости run_config

In [27]:
NUMBERS = {w: str(n) for n, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve".split())}

def norm(text):
    text = str(text).casefold()
    m = re.search(r"<answer>(.*?)</answer>", text, re.S)
    if m: 
        text = m.group(1)
    m = re.search(r"final:\s*(.*)", text)
    if m: 
        text = m.group(1)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"\b(\d+)[-\s]+days?\b", r"\1 day", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(NUMBERS.get(w,w) for w in text.split())

def is_correct(task, answer):
    gold = norm(task["answer"]); pred = norm(answer)
    if not gold or not pred or "not_found" in pred or "ошибка" in pred or "tool_error" in pred: 
        return False
    if re.search(r"\b(not|never|no)\b", pred): 
        return False
    return gold == pred or (" " + gold + " ") in (" " + pred + " ")

RESULTS = globals().get("RESULTS", {})

def report(tags=None):
    rows=[]
    for tag in (tags or RESULTS):
        res=RESULTS[tag]
        calls=[c for c in CALLS if c["tag"]==tag]
        n=len(res); ok=sum(bool(r["ok"]) for r in res)
        costs=[c["cost"] for c in calls]
        measured=(len(calls)>0 and all(v is not None for v in costs))
        cost=sum(costs) if measured else None
        tok=sum((c["prompt"] or 0)+(c["completion"] or 0) for c in calls)
        rows.append({"конфигурация":tag,"верно":f"{ok}/{n}","доля":ok/n if n else np.nan,
         "вызовов на вопрос":len(calls)/n if n else np.nan,"токенов на вопрос":tok/n if n else np.nan,
         "центов на вопрос":cost*100/n if cost is not None and n else np.nan,
         "центов за верный":cost*100/ok if cost is not None and ok else np.nan,
         "вызовов без usage.cost":sum(v is None for v in costs), "ошибок":sum(bool(r.get("error")) for r in res)})
    return pd.DataFrame(rows)


In [28]:
def run_config(tag, fn, sample=None, workers=1):
    sample=tasks if sample is None else sample
    if workers != 1: 
        raise ValueError("Последовательный запуск обязателен для глобальной TAG")
    TAG["now"]=tag
    out_path = DATA / ("run_" + re.sub(r"[^\w-]+", "_", tag.lower()) + ".jsonl")
    saved={}
    if out_path.exists():
        for line in out_path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r=json.loads(line); saved[r["id"]]=r
    rows=[]
    for t in tqdm(sample, desc=tag, unit="вопрос"):
        prev=saved.get(t["id"])
        if prev and prev.get("question")==t["question"] and prev.get("gold")==t["answer"] and not prev.get("error"):
            row=prev.copy()
        else:
            before=len(CALLS)
            try: 
                answer=str(fn(t["question"])); error=""
            except Exception as exc: 
                answer="";error=f"{type(exc).__name__}: {exc}"
            row={"id":t["id"],"question":t["question"],"gold":t["answer"],"answer":answer,
                 "ok":is_correct(t,answer) if not error else False,"error":error,
                 "new_calls":len(CALLS)-before}
            saved[t["id"]]=row
            tmp=out_path.with_suffix(".tmp")
            tmp.write_text("".join(json.dumps(r,ensure_ascii=False)+"\n" for r in saved.values()),encoding="utf-8")
            tmp.replace(out_path)
        rows.append(row)
    RESULTS[tag]=rows
    return report([tag])

def wrong(tag,n=10):
    return pd.DataFrame([{"id":r["id"],"эталон":r["gold"],"ответ":r["answer"],"error":r.get("error","")}
                         for r in RESULTS[tag] if not r["ok"]][:n])

In [29]:
TAG["now"]="счётчик: проба"
print(chat([{"role":"user","content":"Say OK"}])["content"])
display(pd.DataFrame(CALLS).tail(1))

OK!


,tag,model,messages,tools,prompt,completion,cost,status,has_usage
1,счётчик: проба,openai/gpt-4o-mini,1,0,9,2,0.000003,200,True


In [53]:
sample = tasks

display(
    run_config(
        "LangGraph исправленный, mini, test3",
        bare_agent,
        sample=sample
    )
)

display(
    wrong("LangGraph исправленный, mini, test3")
)

LangGraph исправленный, mini, test3:   0%|          | 0/40 [00:00<?, ?вопрос/s]

,конфигурация,верно,доля,вызовов на вопрос,токенов на вопрос,центов на вопрос,центов за верный,вызовов без usage.cost,ошибок
0,"LangGraph исправленный, mini, test3",18/40,0.45,4.05,1324.6,0.023775,0.052833,0,0


,id,эталон,ответ,error
0,fresh-0,15 March,Ukraine boycotted the closing ceremony of the ...,
1,fresh-2,Miguel Díaz-Canel,Iran entered negotiations with the Trump admin...,
2,fresh-4,A total fire ban,I could not find specific information about a ...,
3,fresh-8,Pavel Durov,I could not find specific information regardin...,
4,fresh-15,Oleksandr Syrskyi,The information regarding the dismissal of the...,
5,fresh-23,German Galushchenko,I couldn't find specific information about an ...,
6,fresh-28,nine,Six people were killed in the mass shooting at...,
7,fresh-50,"802,500 lira",No information found.,
8,fresh-54,80,No information found.,
9,fresh-58,BYD Auto,I found no information regarding which car com...,


## Шаг 2. Фреймворк и его налог

Выберу для переноса агента на Langgraph

In [31]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

LC_TOOLS = [tool(web_search), tool(page_find)]
LG_MIDDLEWARE = []

def lc_model(model=CHEAP, temperature=0.0):
    return ChatOpenAI(
        model=model,
        base_url=BASE_URL,
        api_key=KEY,
        http_client=HTTP,
        temperature=temperature,
        timeout=40,
        max_retries=0
    )

In [36]:
def run_langgraph(question, model=CHEAP, temperature=0.0, extra=""):
    agent = create_agent(lc_model(model, temperature), tools=LC_TOOLS, system_prompt=AGENT_SYSTEM + extra, middleware=LG_MIDDLEWARE)
    out = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        {"recursion_limit": 20}
    )
    return out["messages"]


In [33]:
def ask_langgraph(question, model=CHEAP, temperature=0.0, extra=""):
    return run_langgraph(question, model, temperature, extra)[-1].content


In [37]:
TAG["now"] = "проба"
print(ask_langgraph(tasks[0]["question"]), "| эталон:", tasks[0]["answer"])
run_config("LangGraph как есть, mini", ask_langgraph)
wrong("LangGraph как есть, mini", 15)

Ukraine boycotted the closing ceremony of the 2026 Winter Paralympics on March 15, 2026. | эталон: 15 March


LangGraph как есть, mini:   0%|          | 0/40 [00:00<?, ?вопрос/s]

,id,эталон,ответ,error
0,fresh-0,15 March,Ukraine boycotted the closing ceremony of the ...,
1,fresh-2,Miguel Díaz-Canel,Iran entered negotiations with the Trump admin...,
2,fresh-4,A total fire ban,"No specific measure or law from Victoria, Aust...",
3,fresh-7,Deniz Göktaş,No information found.,
4,fresh-15,Oleksandr Syrskyi,,GraphRecursionError: Recursion limit of 20 rea...
5,fresh-18,13 July,I couldn't find the specific date when the Mun...,
6,fresh-23,German Galushchenko,No information found regarding an arrest in Uk...,
7,fresh-28,nine,Six people were killed in the mass shooting at...,
8,fresh-34,6.3,I couldn't find the specific magnitude of the ...,
9,fresh-45,F-16 Fighting Falcon,No information found.,


In [39]:
from langchain.agents.middleware import wrap_model_call
from langchain_core.messages import HumanMessage

class SearchArgs(BaseModel):
    query: str = Field(description="short query: a name, a title, an event")

class FindArgs(BaseModel):
    title: str = Field(description="exact article title, e.g. 2026 in Japan")
    keywords: str = Field(description="two to five keywords")

LC_TOOLS = [tool(web_search, args_schema=SearchArgs), tool(page_find, args_schema=FindArgs)]

In [40]:
@wrap_model_call
def last_step_without_tools(request, handler):
    if sum(m.type == 'ai' for m in request.messages) >= 5:
        request = request.override(tools=[], messages=request.messages + [HumanMessage(LAST_STEP)])
    return handler(request)



In [41]:
if last_step_without_tools not in LG_MIDDLEWARE:
    LG_MIDDLEWARE.append(last_step_without_tools)
run_config("LangGraph, mini", ask_langgraph)

LangGraph, mini:   0%|          | 0/40 [00:00<?, ?вопрос/s]

,конфигурация,верно,доля,вызовов на вопрос,токенов на вопрос,центов на вопрос,центов за верный,вызовов без usage.cost,ошибок
0,"LangGraph, mini",18/40,0.45,0.0,0.0,NaN,NaN,0,0


In [38]:
PROBE = "What is 2 + 2? Reply with the number only."

tags = [
    "пустой вопрос: голый цикл",
    "пустой вопрос: LangGraph"
]

for tag in tags:
    FIRST.pop(tag, None)
    FIRST_RAW.pop(tag, None)

CALLS[:] = [
    c for c in CALLS
    if c["tag"] not in tags
]

TAG["now"] = tags[0]
print("Bare:", bare_agent(PROBE))

TAG["now"] = tags[1]
print("LangGraph:", ask_langgraph(PROBE))

for tag in tags:
    print(
        tag,
        "| JSON:", tag in FIRST,
        "| RAW:", tag in FIRST_RAW,
        "| байт:", len(FIRST_RAW.get(tag, b""))
    )

assert all(tag in FIRST and tag in FIRST_RAW for tag in tags)

Bare: 4
LangGraph: 4
пустой вопрос: голый цикл | JSON: True | RAW: True | байт: 1015
пустой вопрос: LangGraph | JSON: True | RAW: True | байт: 882


In [43]:
comparison = []

for tag in tags:
    hits = [
        c for c in CALLS
        if c["tag"] == tag
    ]

    comparison.append({
        "tag": tag,
        "first_request_bytes": len(FIRST_RAW[tag]),
        "first_input_tokens": hits[0]["prompt"] if hits else None,
        "first_tools": len(FIRST[tag].get("tools", [])),
        "first_messages": len(FIRST[tag].get("messages", []))
    })

    filename = (
        "first_request_bare.json"
        if "голый цикл" in tag
        else "first_request_langgraph.json"
    )

    (DATA / filename).write_text(
        json.dumps(FIRST[tag], ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

display(pd.DataFrame(comparison))

print(
    "Byte-identical:",
    FIRST_RAW[tags[0]] == FIRST_RAW[tags[1]]
)

a = json.dumps(
    FIRST[tags[0]], ensure_ascii=False, indent=2
).splitlines()

b = json.dumps(
    FIRST[tags[1]], ensure_ascii=False, indent=2
).splitlines()

print("\n".join(
    list(difflib.unified_diff(
        a, b,
        fromfile="bare",
        tofile="langgraph"
    ))[:140]
))

,tag,first_request_bytes,first_input_tokens,first_tools,first_messages
0,пустой вопрос: голый цикл,1015,168,2,2
1,пустой вопрос: LangGraph,882,136,2,2


Byte-identical: False
--- bare

+++ langgraph

@@ -1,13 +1,14 @@

 {
   "model": "openai/gpt-4o-mini",
+  "stream": false,
   "messages": [
     {
-      "role": "system",
-      "content": "You answer questions about events of 2026 using Wikipedia tools. Never answer from memory, do not repeat the same call. When done, reply with the short answer only, no explanation."
+      "content": "You answer questions about events of 2026 using Wikipedia tools. Never answer from memory, do not repeat the same call. When done, reply with the short answer only, no explanation.",
+      "role": "system"
     },
     {
-      "role": "user",
-      "content": "What is 2 + 2? Reply with the number only."
+      "content": "What is 2 + 2? Reply with the number only.",
+      "role": "user"
     }
   ],
   "temperature": 0.0,
@@ -18,16 +19,15 @@

         "name": "web_search",
         "description": "Search English Wikipedia: returns the intro of the best article and the titles of two more articles."

In [44]:
framework_tax = {
    "bare_bytes": len(FIRST_RAW["пустой вопрос: голый цикл"]),
    "langgraph_bytes": len(FIRST_RAW["пустой вопрос: LangGraph"]),
}

framework_tax["delta_bytes"] = (
    framework_tax["langgraph_bytes"]
    - framework_tax["bare_bytes"]
)

framework_tax["delta_percent"] = (
    framework_tax["delta_bytes"]
    / framework_tax["bare_bytes"] * 100
)

print(framework_tax)

(DATA / "framework_tax.json").write_text(
    json.dumps(framework_tax, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

{'bare_bytes': 1015, 'langgraph_bytes': 882, 'delta_bytes': -133, 'delta_percent': -13.10344827586207}


112

## Шаг 3. ДВа паттерна недели

Вначале реализуем критика как на семинаре

In [47]:
from langgraph.graph import StateGraph, START, END

class QA(TypedDict):
    question: str
    answer: str
    evidence: str
    feedback: str
    rounds: int
    checks: int

class Verdict(BaseModel):
    supported: bool = Field(description="true only if the evidence explicitly supports the answer")
    issue: str = Field(description="what is wrong or missing, empty if supported")

CRITIC_SYSTEM = ("You check a short answer against the evidence the agent retrieved from Wikipedia. Judge only by the evidence. "
                 "If the evidence does not contain the answer, the answer is not supported.")

def answer_node(state):
    print(
        f"[ANSWER] round={state['rounds'] + 1}",
        f"question={state['question'][:70]}",
        flush=True
    )
    note = ""
    if state["feedback"]:
        note = f"\n\nA reviewer rejected your previous answer '{state['answer']}': {state['feedback']} Search again and correct it."
    messages = run_langgraph(state["question"] + note)
    found = "\n".join(str(m.content)[:800] for m in messages if m.type == "tool")
    return {"answer": messages[-1].content, "evidence": found[-4000:], "rounds": state["rounds"] + 1}

In [46]:
def critic_node(state):
    print(
        f"[CRITIC] check={state['checks'] + 1}",
        flush=True
    )
    judge = lc_model(CHEAP).with_structured_output(Verdict)
    verdict = judge.invoke([
        ("system", CRITIC_SYSTEM),
        ("user", f"Question: {state['question']}\nAnswer: {state['answer']}\n"
                 f"Evidence the agent found:\n{state['evidence'] or 'nothing'}")
    ])
    return {"feedback": "" if verdict.supported else verdict.issue, "checks": state["checks"] + 1}


In [48]:
def route(state):
    if state["feedback"] and state["rounds"] < 2:
        return 'answer'
    return END


In [49]:
graph = StateGraph(QA)
graph.add_node("answer", answer_node)
graph.add_node("critic", critic_node)
graph.add_edge(START, "answer")
graph.add_edge("answer", "critic")
graph.add_conditional_edges("critic", route, ["answer", END])
checked = graph.compile()
print(checked.get_graph().draw_mermaid())

def ask_checked(question):
    return checked.invoke({"question": question, "answer": "", "evidence": "", "feedback": "", "rounds": 0, "checks": 0})["answer"]

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	answer(answer)
	critic(critic)
	__end__([<p>__end__</p>]):::last
	__start__ --> answer;
	answer --> critic;
	critic -.-> __end__;
	critic -.-> answer;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [54]:
display(run_config("LangGraph + критик, mini",ask_checked,sample=sample))


LangGraph + критик, mini:   0%|          | 0/40 [00:00<?, ?вопрос/s]

[ANSWER] round=1 question=What measure came into force in Victoria, Australia in January 2026?
[CRITIC] check=1
[ANSWER] round=1 question=How many people were killed in the car incident in Leipzig, Germany in
[CRITIC] check=1
[ANSWER] round=1 question=Who was imprisoned in Istanbul for insulting President Erdoğan in July
[CRITIC] check=1
[ANSWER] round=2 question=Who was imprisoned in Istanbul for insulting President Erdoğan in July
[CRITIC] check=2
[ANSWER] round=1 question=Who was charged by the FSB in Russia on 29 July 2026?
[CRITIC] check=1
[ANSWER] round=1 question=What city hosted the 2026 BWF World Championships in sports in August?
[CRITIC] check=1
[ANSWER] round=1 question=How many people were killed in the February 18 explosion at a firework
[CRITIC] check=1
[ANSWER] round=1 question=Who was dismissed as Commander-in-Chief of the Armed Forces of Ukraine
[CRITIC] check=1
[ANSWER] round=2 question=Who was dismissed as Commander-in-Chief of the Armed Forces of Ukraine
[CRITIC] c

,конфигурация,верно,доля,вызовов на вопрос,токенов на вопрос,центов на вопрос,центов за верный,вызовов без usage.cost,ошибок
0,"LangGraph + критик, mini",19/40,0.475,8.275,2837.95,0.05033,0.105958,0,0


In [55]:
a={r["id"]:r for r in RESULTS["LangGraph, mini"]}
b={r["id"]:r for r in RESULTS["LangGraph + критик, mini"]}
shared=sorted(set(a)&set(b))
fixed=sum(not a[i]["ok"] and b[i]["ok"] for i in shared)
broken=sum(a[i]["ok"] and not b[i]["ok"] for i in shared)
print("Критик исправил:",fixed,"испортил:",broken,"общих ID:",len(shared))
(DATA/"critic_counts.json").write_text(json.dumps({"fixed":fixed,"broken":broken,"n":len(shared)},indent=2),encoding="utf-8")

Критик исправил: 3 испортил: 2 общих ID: 40


42

In [56]:
PLAN_SYSTEM = ("Write a short numbered plan (two to four steps) to answer the question with two Wikipedia tools: web_search finds "
               "an article, page_find looks for keywords inside an article. Note that the dataset contain qustion about 2026+ period of time and you can unknown all this fact. Just try to make a plan no find some facts. Plan only, no answer.")

In [57]:
def plan_then_act(question):
    plan = lc_model(CHEAP).invoke([("system", PLAN_SYSTEM), ("user", question)]).content
    return ask_langgraph(question, CHEAP, extra="\nFollow this plan:\n" + plan)


In [58]:
display(run_config("план + LangGraph, mini",plan_then_act,sample=sample))
display(report(["LangGraph, mini","LangGraph + критик, mini","план + LangGraph, mini"]))

план + LangGraph, mini:   0%|          | 0/40 [00:00<?, ?вопрос/s]

,конфигурация,верно,доля,вызовов на вопрос,токенов на вопрос,центов на вопрос,центов за верный,вызовов без usage.cost,ошибок
0,"план + LangGraph, mini",5/40,0.125,5.6,1943.675,0.036969,0.295755,0,0


,конфигурация,верно,доля,вызовов на вопрос,токенов на вопрос,центов на вопрос,центов за верный,вызовов без usage.cost,ошибок
0,"LangGraph, mini",18/40,0.450,0.000,0.000,NaN,NaN,0,0
1,"LangGraph + критик, mini",19/40,0.475,8.275,2837.950,0.050330,0.105958,0,0
2,"план + LangGraph, mini",5/40,0.125,5.600,1943.675,0.036969,0.295755,0,0


Добавим рассмотрение голосование и каскада

In [59]:
def bag(answer):
    return " ".join(sorted(norm(answer).split()))

In [60]:
def vote(question, k=3):
    with ThreadPoolExecutor(k) as ex:
        answers = list(ex.map(lambda _: ask_langgraph(question, CHEAP, 0.7), range(k)))
    keys = [bag(a) for a in answers]
    top, n = Counter(keys).most_common(1)[0]
    return {"answer": answers[keys.index(top)], "agreement": n / k}


In [61]:
def cascade(question):
    v = vote(question, 3)
    if v["agreement"] == 1:
        return v["answer"]
    ESCALATED.append(question)
    return ask_langgraph(question, STRONG)

In [62]:
RUN_BONUS_VOTE=False
ESCALATED=[]
if KEY and RUN_BONUS_VOTE:
    display(run_config("голосование 3, mini",lambda q: vote(q,3)["answer"],sample=sample))
    display(run_config("каскад: 3 голоса mini, иначе Sonnet",cascade,sample=sample))
    print("Эскалаций:",len(ESCALATED),"из",len(sample),"доля:",len(ESCALATED)/len(sample))


## Шаг 4.  Итог

In [63]:
def strong_alone(question):
    msg=chat([{"role":"system","content":"Answer briefly from your own knowledge. If unknown, say NOT_FOUND. No tools."},
              {"role":"user","content":question}],model=STRONG)
    return msg.get("content") or ""


In [64]:
display(run_config("Sonnet без инструментов", strong_alone, sample=sample))

Sonnet без инструментов:   0%|          | 0/40 [00:00<?, ?вопрос/s]

,конфигурация,верно,доля,вызовов на вопрос,токенов на вопрос,центов на вопрос,центов за верный,вызовов без usage.cost,ошибок
0,Sonnet без инструментов,0/40,0.0,1.0,87.75,0.075135,NaN,0,0


In [65]:
AUDIT_PATH=DATA / "judge_audit.csv"
audit=[]
for tag, rows in RESULTS.items():
    wrong_rows=[r for r in rows if not r["ok"]][:10]
    for r in wrong_rows:
        audit.append({"config":tag,"id":r["id"],"question":r["question"],"gold":r["gold"],
                     "answer":r["answer"],"auto_correct":r["ok"],"human_correct":"","reason":""})
if audit:
    pd.DataFrame(audit).to_csv(AUDIT_PATH,index=False)
    display(pd.DataFrame(audit).head(10))
    print("Заполни human_correct=true/false и reason в",AUDIT_PATH)


,config,id,question,gold,answer,auto_correct,human_correct,reason
0,"LangGraph исправленный, mini, test3",fresh-0,On what date did Ukraine boycott the closing c...,15 March,Ukraine boycotted the closing ceremony of the ...,False,,
1,"LangGraph исправленный, mini, test3",fresh-2,Who entered negotiations with the Trump admini...,Miguel Díaz-Canel,Iran entered negotiations with the Trump admin...,False,,
2,"LangGraph исправленный, mini, test3",fresh-4,"What measure came into force in Victoria, Aust...",A total fire ban,I could not find specific information about a ...,False,,
3,"LangGraph исправленный, mini, test3",fresh-8,Who was charged by the FSB in Russia on 29 Jul...,Pavel Durov,I could not find specific information regardin...,False,,
4,"LangGraph исправленный, mini, test3",fresh-15,Who was dismissed as Commander-in-Chief of the...,Oleksandr Syrskyi,The information regarding the dismissal of the...,False,,
5,"LangGraph исправленный, mini, test3",fresh-23,"In Ukraine, who was arrested on February 15, 2...",German Galushchenko,I couldn't find specific information about an ...,False,,
6,"LangGraph исправленный, mini, test3",fresh-28,What was the number of people killed in the ma...,nine,Six people were killed in the mass shooting at...,False,,
7,"LangGraph исправленный, mini, test3",fresh-50,What fine was issued to Amed S.F.K. by the Tur...,"802,500 lira",No information found.,False,,
8,"LangGraph исправленный, mini, test3",fresh-54,How many people were killed in the fire at a s...,80,No information found.,False,,
9,"LangGraph исправленный, mini, test3",fresh-58,Which car company overtook Tesla as the world'...,BYD Auto,I found no information regarding which car com...,False,,


Заполни human_correct=true/false и reason в C:\Users\alex-\Documents\AI-agent\pythonProject4\hw_3\data\judge_audit.csv


In [66]:
def color_of(tag):
    return COLORS["red"] if "каскад" in tag else COLORS["amber"] if "Sonnet" in tag else COLORS["teal"] if "smolagents" in tag else COLORS["violet"]

def money_chart(tags, path, title):
    table = report(tags)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.set_xscale("log")
    for _, r in table.iterrows():
        ax.scatter(r["центов на вопрос"], r["доля"] * 100, s=110, color=color_of(r["конфигурация"]), zorder=3)
    ax.margins(x=0.15, y=0.12)
    fig.canvas.draw()
    placed = []
    for _, r in table.sort_values("центов на вопрос").iterrows():
        for dx, dy in [(8, 4), (8, -12), (-8, 4), (-8, -12), (8, 16), (8, -24)]:
            label = ax.annotate(r["конфигурация"], (r["центов на вопрос"], r["доля"] * 100), xytext=(dx, dy), textcoords="offset points",
                                fontsize=8.5, ha="left" if dx > 0 else "right", color=color_of(r["конфигурация"]))
            box = label.get_window_extent()
            if not any(box.overlaps(b) for b in placed) or dy == -24:
                break
            label.remove()
        placed.append(box)
    ax.set_xlabel("цена вопроса, центы (логарифмическая шкала)"); ax.set_ylabel("доля верных, %")
    ax.set_title(title); ax.grid(alpha=0.3)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return table



In [67]:
REQUIRED=["голый цикл, mini","LangGraph, mini","LangGraph + критик, mini","план + LangGraph, mini","Sonnet без инструментов"]
if all(tag in RESULTS for tag in REQUIRED):
    final_table=report(REQUIRED)
    final_table.to_csv(DATA/"report.csv",index=False)
    display(final_table)
    if final_table["центов на вопрос"].notna().all() and (final_table["центов на вопрос"]>0).all():
        display(money_chart(REQUIRED,str(IMG/"money_chart.png"),"ДЗ 3: цена вопроса против доли верных"))
    else:
        print("Не строю логарифмическую ось: отсутствует usage.cost или 0 цена")
else:
    print("Запусти пять обязательных конфигураций на одном sample")


Запусти пять обязательных конфигураций на одном sample


## Шаг 5. Лимит на число модельных вызовов

In [68]:
from langchain.agents.middleware import ModelCallLimitMiddleware

def capped_demo(question, limit=2):
    capped=create_agent(lc_model(CHEAP),tools=LC_TOOLS,system_prompt=AGENT_SYSTEM,
                        middleware=[ModelCallLimitMiddleware(run_limit=limit,exit_behavior="end")])
    TAG["now"]="бюджет: демонстрация"
    start=len(CALLS)
    out=capped.invoke({"messages":[{"role":"user","content":question}]})
    answer=str(out["messages"][-1].content)
    calls=len(CALLS)-start
    assert calls<=limit, f"Вызовов {calls} > {limit}"
    return {"question":question,"answer":answer,"model_calls":calls,"limit":limit}


capped_result=capped_demo("Compare the dates of the 2026 Winter Olympics and the 2026 FIFA World Cup, then list three events of each.")
display(capped_result)
(DATA/"call_limit_demo.json").write_text(json.dumps(capped_result,ensure_ascii=False,indent=2),encoding="utf-8")


{'question': 'Compare the dates of the 2026 Winter Olympics and the 2026 FIFA World Cup, then list three events of each.',
 'answer': 'Model call limits exceeded: run limit (2/2)',
 'model_calls': 2,
 'limit': 2}

219